# FOLIO Two-Stage: 41 Verified-But-Wrong Cases Analysis

Comparing Two-Stage error patterns with simple Lean to understand if iteration helps.

In [ ]:
import json
import pandas as pd

# Load results
with open('../results/folio/lean/all_results.json', 'r') as f:
    lean_results = json.load(f)

with open('../results/folio/two_stage/all_results.json', 'r') as f:
    two_stage_results = json.load(f)

# Get verified-but-wrong cases
lean_vw = [r for r in lean_results if r.get('lean_verification', {}).get('success', False) and not r['correct']]

two_stage_vw = []
for r in two_stage_results:
    if 'all_cycles' in r and r['all_cycles']:
        if r['all_cycles'][-1].get('lean_success', False) and not r['correct']:
            two_stage_vw.append(r)

print(f"Lean verified-but-wrong: {len(lean_vw)} cases")
print(f"Two-Stage verified-but-wrong: {len(two_stage_vw)} cases")
print(f"\nTwo-Stage has {len(lean_vw) - len(two_stage_vw)} fewer errors")

## Pattern Comparison

In [ ]:
# Count patterns for both
def get_patterns(cases):
    patterns = {}
    for case in cases:
        pattern = f"{case.get('ground_truth')} → {case.get('prediction')}"
        patterns[pattern] = patterns.get(pattern, 0) + 1
    return patterns

lean_patterns = get_patterns(lean_vw)
ts_patterns = get_patterns(two_stage_vw)

# Create comparison table
all_patterns = set(lean_patterns.keys()) | set(ts_patterns.keys())
comparison = []
for pattern in sorted(all_patterns):
    lean_count = lean_patterns.get(pattern, 0)
    ts_count = ts_patterns.get(pattern, 0)
    comparison.append({
        'Pattern': pattern,
        'Lean': lean_count,
        'Two-Stage': ts_count,
        'Difference': lean_count - ts_count
    })

df = pd.DataFrame(comparison)
print("\nPattern Comparison:")
print("="*70)
print(df.to_string(index=False))
print("\nKey Observations:")
print(f"- False → True: Lean {lean_patterns.get('False → True', 0)}, Two-Stage {ts_patterns.get('False → True', 0)}")
print(f"  Two-stage reduces this by {lean_patterns.get('False → True', 0) - ts_patterns.get('False → True', 0)} cases!")
print(f"- Uncertain → True: Lean {lean_patterns.get('Uncertain → True', 0)}, Two-Stage {ts_patterns.get('Unknown → True', 0)}")
print(f"  Note: 'Uncertain' vs 'Unknown' are same ground truth")

## Pattern 1: Unknown → True (12 cases, 29.3%)

Same as Lean's "Uncertain → True" - axiomatizing unmentioned conclusions.

**The Problem**: Model axiomatizes conclusions about facts NOT in premises.

### Example 1.2: ID 2 - Joey the Turkey

**Premises**: All about TOM (wild turkeys, Tom is a wild turkey, etc.)

**Conclusion**: "Joey is a wild turkey"

**Ground Truth**: Unknown (Joey never mentioned!)

**Model Prediction**: True

**Problem in Lean Code**:
```lean
axiom Joey : Entity
axiom Joey_is_Wild : WildTurkey Joey  <-- PROBLEM!
```

**Analysis**: Same problem as simple Lean. Model declares Joey exists, then axiomatizes Joey_is_Wild. But Joey is never mentioned in premises! Two-stage iteration didn't fix this axiomatization issue.

In [ ]:
# Example 1014 - Bonnie
ex1014 = next((r for r in two_stage_vw if r.get('example_id') == 1014), None)

if ex1014:
    print("Example 1014 (Bonnie) - Two-Stage:")
    print("="*70)
    print(f"Premises: {ex1014.get('premises')[:200]}...")
    print(f"\nConclusion: {ex1014.get('conclusion')}")
    print(f"Ground truth: {ex1014.get('ground_truth')} (not in premises)")
    print(f"Prediction: {ex1014.get('prediction')}")
    print(f"Num cycles: {len(ex1014['all_cycles'])}")

    final_cycle = ex1014['all_cycles'][-1]
    code = final_cycle.get('lean_code', '')

    print("\nKey lines in final Lean code:")
    for i, line in enumerate(code.split('\n'), 1):
        if i >= 20 and i <= 30:
            marker = '  <-- PROBLEM!' if i == 24 else ''
            print(f"{i:3d}: {line}{marker}")

    print("\nANALYSIS:")
    print("Line 23 comment: 'Given in Stage 1 reasoning: Bonnie performs'")
    print("Line 24 axiomatizes bonnie_performs - but this is the CONCLUSION!")
    print("Premises don't say Bonnie performs. Should be Unknown.")

### Example 1.1: ID 1014 - Bonnie Performs

**Premises**: Rules about people in club (perform OR inactive, perform → engaged, chaperone → not student)

**Conclusion**: "Bonnie performs in school talent shows often"

**Ground Truth**: Unknown (premises don't say Bonnie performs!)

**Model Prediction**: True

**Problem in Lean Code**:
```lean
-- Line 23: Comment says "Given in Stage 1 reasoning"
-- Line 24: axiom bonnie_performs : PerformsOften Bonnie  <-- PROBLEM!
```

**Analysis**: The comment at Line 23 reveals the issue: "Given in Stage 1 reasoning: Bonnie performs in school talent shows often". This is the CONCLUSION we're trying to prove! The model axiomatizes it based on Stage 1's natural language reasoning, then uses it to prove Bonnie is a student. But the premises never say Bonnie performs - this should be Unknown.

In [ ]:
# Example 2 - Joey
ex2 = next(r for r in two_stage_vw if r.get('example_id') == 2)

print("Example 2 (Joey) - Two-Stage:")
print("="*70)
print(f"Premises: {ex2.get('premises')[:150]}...")
print(f"\nConclusion: {ex2.get('conclusion')}")
print(f"Ground truth: Unknown (not in premises)")
print(f"Prediction: {ex2.get('prediction')}")
print(f"Num cycles: {len(ex2['all_cycles'])}")

final_cycle = ex2['all_cycles'][-1]
code = final_cycle.get('lean_code', '')
print("\nKey lines in final Lean code:")
for i, line in enumerate(code.split('\n'), 1):
    if 'Joey' in line and i <= 31:
        marker = '  <-- PROBLEM!' if 'axiom' in line and 'W' in line else ''
        print(f"{i:3d}: {line}{marker}")

print("\nANALYSIS:")
print("Line 30 axiomatizes W_Joey (Joey is wild turkey)")
print("Same problem as simple Lean - two-stage iteration didn't fix it!")

## Pattern 2: True/False → Unknown (19 cases, 46.3%)

Combined: True → Unknown (10) + False → Unknown (9)

Model fails to derive valid conclusions or contradictions.

In [ ]:
# True → Unknown cases
true_unknown = [r for r in two_stage_vw if r.get('ground_truth') == 'True' and r.get('prediction') == 'Unknown']
false_unknown = [r for r in two_stage_vw if r.get('ground_truth') == 'False' and r.get('prediction') == 'Unknown']

print(f"True → Unknown: {len(true_unknown)} cases")
print(f"False → Unknown: {len(false_unknown)} cases")
print(f"Total → Unknown: {len(true_unknown) + len(false_unknown)} cases (46.3%)")
print("\nThese are reasoning failures, not axiomatization problems")
print("Model gives up and says Unknown when answer is actually derivable")

## Pattern 3: False → True (4 cases, 9.8%)

Much better than simple Lean's 21 cases!

In [ ]:
false_true = [r for r in two_stage_vw if r.get('ground_truth') == 'False' and r.get('prediction') == 'True']

print(f"False → True cases in Two-Stage: {len(false_true)}")
print(f"False → True cases in simple Lean: {len([r for r in lean_vw if r.get('ground_truth') == 'False' and r.get('prediction') == 'True'])}")
print(f"\nImprovement: Two-stage reduces this by 17 cases (81% reduction!)")
print("\nRemaining cases:")
for case in false_true:
    print(f"  ID {case.get('example_id')}: {case.get('conclusion')[:50]}...")

## Pattern 4: Unknown → False (4 cases, 9.8%)

In [ ]:
unknown_false = [r for r in two_stage_vw if r.get('ground_truth') in ['Unknown', 'Uncertain'] and r.get('prediction') == 'False']

print(f"Unknown → False cases: {len(unknown_false)}")
for case in unknown_false:
    print(f"  ID {case.get('example_id')}: {case.get('conclusion')[:50]}...")

## Pattern 5: True → False (2 cases, 4.9%)

In [ ]:
true_false = [r for r in two_stage_vw if r.get('ground_truth') == 'True' and r.get('prediction') == 'False']

print(f"True → False cases: {len(true_false)}")
for case in true_false:
    print(f"  ID {case.get('example_id')}: {case.get('conclusion')[:50]}...")

## Which Cases Did Two-Stage Fix?

In [ ]:
# Find cases that are wrong in Lean but correct in Two-Stage
lean_wrong_ids = set(r.get('example_id') for r in lean_vw)
ts_wrong_ids = set(r.get('example_id') for r in two_stage_vw)

fixed_by_ts = lean_wrong_ids - ts_wrong_ids

print(f"Cases fixed by Two-Stage: {len(fixed_by_ts)}")
print("\nThese are cases where simple Lean was verified-but-wrong,")
print("but Two-Stage got the right answer.")
print("\nFixed case IDs:")
print(sorted(fixed_by_ts)[:20])  # First 20

## Summary: Key Findings

In [ ]:
print("="*70)
print("SUMMARY: TWO-STAGE vs SIMPLE LEAN")
print("="*70)

print(f"\nVerified-but-wrong cases:")
print(f"  Simple Lean: {len(lean_vw)} cases")
print(f"  Two-Stage:   {len(two_stage_vw)} cases")
print(f"  Improvement: {len(lean_vw) - len(two_stage_vw)} fewer errors ({(len(lean_vw) - len(two_stage_vw))/len(lean_vw)*100:.1f}% reduction)")

print(f"\nPattern Comparison:")
print(f"\n1. False → True (Axiomatizing contradictions):")
print(f"   Lean: 21 cases (42.9%)")
print(f"   Two-Stage: 4 cases (9.8%)")
print(f"   ✓ Two-stage fixes 17 cases (81% improvement!)")

print(f"\n2. Unknown → True (Axiomatizing unmentioned facts):")
print(f"   Lean: 18 cases (36.7%)")
print(f"   Two-Stage: 12 cases (29.3%)")
print(f"   ✓ Two-stage fixes 6 cases (33% improvement)")

print(f"\n3. → Unknown (Reasoning failures):")
print(f"   Lean: 9 cases (18.4%)")
print(f"   Two-Stage: 19 cases (46.3%)")
print(f"   ✗ Two-stage has MORE reasoning failures")

print(f"\n" + "="*70)
print("KEY INSIGHTS:")
print("="*70)
print("\n1. Two-stage DOES help with axiomatization problems")
print("   - 81% reduction in False → True errors")
print("   - Stage 1 reasoning catches some contradictions")

print("\n2. But it doesn't solve the problem completely")
print("   - Still axiomatizes unmentioned conclusions (12 cases)")
print("   - Example: Joey case still wrong in two-stage")

print("\n3. Two-stage trades axiomatization for reasoning failures")
print("   - Fewer False → True errors (good)")
print("   - More → Unknown errors (model gives up more often)")

print("\n4. Net effect: 8 fewer verified-but-wrong cases (16% improvement)")
print("   - Not a huge gain considering the extra complexity")
print("="*70)